## Phần 0 — Khởi động (5 phút)

- [ ] **0.1 Seed và device.** Viết hàm `seed_everything(seed)` đặt seed cho `random`, `numpy`, `torch` (CPU + CUDA). Tạo biến `device` tự chọn `cuda` nếu có, không thì `cpu`.
  - **Kiểm tra:** gọi `seed_everything(42)` hai lần, mỗi lần in `torch.rand(3)` → hai kết quả giống hệt nhau.

<details><summary>Gợi ý cú pháp</summary>

`random.seed`, `np.random.seed`, `torch.manual_seed`, `torch.cuda.manual_seed_all`, `torch.cuda.is_available()`

In [ ]:
import numpy as np
import torch
import random
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision.transforms import ToTensor

In [2]:
def seed_everything(seed: int):
    ''' seed python before
    seed random lib in python
    seed numpy 
    seed torch
    seed cuda 
    seed compute in GPU
    ** all things in python - becuz it is independence
    =====================
    params: seed (int)
    return device - seed '''

    os.environ['PYTHONASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)

    if torch.cuda.is_available():
        device = torch.device('cuda')
        torch.cuda.seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
    else:
        device = torch.device('cpu')
    print(torch.rand((5, 4)))
    print(device)
    return device
    

In [3]:
seed_everything(42)

tensor([[0.8823, 0.9150, 0.3829, 0.9593],
        [0.3904, 0.6009, 0.2566, 0.7936],
        [0.9408, 0.1332, 0.9346, 0.5936],
        [0.8694, 0.5677, 0.7411, 0.4294],
        [0.8854, 0.5739, 0.2666, 0.6274]])
mps


device(type='mps')

**1.1 Đọc nhãn.** Đọc `train.csv` bằng pandas. In số ảnh, số lớp, số ảnh mỗi lớp. Tạo `class2idx` và `idx2class`.

- **Kiểm tra:** `len(class2idx) == 6`; `idx2class[class2idx['glass']] == 'glass'`.

In [7]:
train_dir = '/Users/quangmanh/Documents/Olympic-AI-Practice-1/đề code tay/data/data-2/dataset/train/annotations.csv'
df = pd.read_csv(train_dir)
print(df.shape)
df.head()

(8385, 3)


,file_name,is_negative,category_id
0,201508010000_1513_8f8e20b8.jpg,False,1.0
1,201508010000_1513_9d8a4002.jpg,True,NaN
2,201508010300_1513_14239637.jpg,False,1.0
3,201508010300_1513_c7793163.jpg,True,NaN
4,201508010600_1513_a80d2e75.jpg,False,2.0


In [18]:
df_neg = df[df['is_negative'] == False]
df.shape[0] - df_neg.shape[0]

2548

In [23]:
df['category_id'].value_counts(dropna=False)

category_id
NaN    2548
1.0    1931
2.0    1583
4.0    1553
3.0     770
Name: count, dtype: int64

In [25]:
CLASSES = ['no_storm', 'cat1', 'cat2', 'cat3', 'cat4']

In [28]:
{ idx: cls_name for idx, cls_name in enumerate(CLASSES)}
{ cls_name: idx for idx, cls_name in enumerate(CLASSES)}

{'no_storm': 0, 'cat1': 1, 'cat2': 2, 'cat3': 3, 'cat4': 4}

In [29]:
df['label'] = df['category_id'].fillna(0).astype(int)
df.head()

,file_name,is_negative,category_id,label
0,201508010000_1513_8f8e20b8.jpg,False,1.0,1
1,201508010000_1513_9d8a4002.jpg,True,NaN,0
2,201508010300_1513_14239637.jpg,False,1.0,1
3,201508010300_1513_c7793163.jpg,True,NaN,0
4,201508010600_1513_a80d2e75.jpg,False,2.0,2


In [37]:
df['storm_id'] = df['file_name'].str.split('_').str[1]
df['year'] = df['storm_id'].str.slice(0, 2)
df.head()

,file_name,is_negative,category_id,label,storm_id,year
0,201508010000_1513_8f8e20b8.jpg,False,1.0,1,1513,15
1,201508010000_1513_9d8a4002.jpg,True,NaN,0,1513,15
2,201508010300_1513_14239637.jpg,False,1.0,1,1513,15
3,201508010300_1513_c7793163.jpg,True,NaN,0,1513,15
4,201508010600_1513_a80d2e75.jpg,False,2.0,2,1513,15


In [41]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
tr_idx, val_idx = next(sgkf.split(df, df.label, groups=df.storm_id))

In [42]:
train_split, val_split = train_test_split(df, random_state=42, stratify=df.label)

In [52]:
class ImageDataset(Dataset):
    def __init__(self, df, img_dir, transform = False, has_label=True):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.has_label = has_label
        self.file_name = self.df.iloc[:,0]
        if self.has_label:
            self.labels = self.df.iloc[:,3]
    def __len__(self):
        return len(self.df)
    def __getitem__(self, index):
        img_name = self.file_name[index]
        img_path = os.join(self.img_dir, img_name)

        image = Image.open(img_path).convert('RGB').resize((224,224))
        image.save(os.join('/Users/quangmanh/Documents/Olympic-AI-Practice-1/đề code tay/data/data-2/dataset/train/images_resize', img_name))

        if self.transform:
            image = self.transform(image)
        else:
            image = ToTensor()(image)
        if self.has_label:
            label = self.labels[index]
            return image, torch.tensor(label, dtype=torch.long)
        return image, img_name

In [ ]:
def make_test_df(img_dir):
    valid_format = ('.png', '.jpg', '.jpeg')

    list_image = sorted(os.listdir(img_dir))

    img_name = [
        f for f in list_image if f.lower().endswith(valid_format) and os.path.isfile(os.path.join(img_dir, f))
    ]

    test_df = pd.DataFrame({'file_name': img_name})
    return test_df

In [53]:
img_dir = '/Users/quangmanh/Documents/Olympic-AI-Practice-1/đề code tay/data/data-2/dataset/train/images'
train_dataset = ImageDataset(df, img_dir=img_dir, has_label=True)

In [54]:
dl_tr = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True, pin_memory=True)